In [ ]:
# Базовые библиотеки для воспроизводимости, анализа и визуализации.
import random
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)
pd.set_option("display.precision", 4)

In [ ]:
# Фиксируем seed и определяем устройство.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
torch.version.__version__

In [ ]:
# Загрузка датасета
emotions = load_dataset("dair-ai/emotion")

# Посмотрим на структуру
print(emotions)

In [ ]:
import pandas as pd

# 1. Переводим в формат Pandas (только для чтения/анализа)
emotions.set_format(type="pandas")
df_train = emotions["train"][:]

# 2. Адаптированный код
print("Размер датасета:", len(df_train))

# В стандартном датасете колонка называется 'label' (ID)
# Чтобы получить названия, используем метаданные датасета
label_names = emotions["train"].features["label"].names

# Создаем словари сопоставления
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}

# Добавляем текстовую колонку для наглядности (наоборот вашему коду)
df_train["Emotion"] = df_train["label"].map(id2label)

print("label2id:", label2id)
print("id2label:", id2label)

# Вывод распределения эмоций
display(df_train["Emotion"].value_counts())

# Пример случайных строк
display(df_train.sample(6, random_state=42).reset_index(drop=True))

# 3. Сбрасываем формат обратно в формат Hugging Face для обучения
emotions.reset_format()

In [ ]:
# Переименуем колонку, чтобы она соответствовала стандартам моделей
emotions = emotions.rename_column("label", "labels")

print("Пример из train:")
# Используем имя переменной 'emotions'
display(pd.DataFrame(emotions["train"][:3]))

print("\nПример из validation:")
display(pd.DataFrame(emotions["validation"][:3]))

# Описание эмоциональных состояний
### В рамках данного датасета мы классифицируем шесть базовых типов эмоциональных состояний:

Sadness (Грусть): Выражение печали, потери, разочарования или подавленности.

Joy (Радость): Проявление позитивного настроя, счастья, успеха или оптимизма.

Love (Любовь): Выражение привязанности, нежности, восхищения или близости.

Anger (Гнев): Выражение агрессии, раздражения или сильного недовольства.

Fear (Страх): Ощущение тревоги, неуверенности, угрозы или паники.

Surprise (Удивление): Реакция на неожиданные события, изумление или шок.

In [ ]:
from transformers import AutoTokenizer
from typing import Dict, List

# 1. Загружаем токенизатор
MODEL_NAME = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", tokenizer.__class__.__name__)
print("Model checkpoint:", MODEL_NAME)

# 2. Функция токенизации (исправлено "Comment" -> "text")
def tokenize_batch(batch: Dict[str, List[str]]) -> Dict[str, any]:
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

# 3. Применяем токенизацию (используем ваше имя переменной 'emotions')
tokenized_datasets = emotions.map(tokenize_batch, batched=True)

# 4. Удаляем текстовый столбец, оставляя только числа (тензоры)
# В этом датасете колонка называется "text"
tokenized_datasets = tokenized_datasets.remove_columns(["text"])

# Проверяем результат
print(tokenized_datasets)

In [ ]:
# Выбираем 3 примера для наглядности
for i in range(3):
    # Берем данные из токенизированного набора и исходный текст
    example = tokenized_datasets["train"][i]

    # ИСПРАВЛЕНО: используем emotions вместо dataset_dict и "text" вместо "Comment"
    raw_text = emotions["train"][i]["text"]

    print(f"{'='*20} РАЗБОР ПРИМЕРА №{i+1} {'='*20}")
    print(f"ИСХОДНЫЙ ТЕКСТ: {raw_text}")

    # 1. ТОКЕНЫ (превращаем ID обратно в слова/субслова)
    tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
    print(f"\n1. ТОКЕНЫ: {tokens[:15]}...")

    # 2. INPUT_IDS (числовые индексы в словаре)
    print(f"2. INPUT_IDS: {example['input_ids'][:15]}...")

    # 3. ATTENTION_MASK (показывает модели, куда смотреть)
    print(f"3. ATTENTION_MASK: {example['attention_mask'][:15]}...")

    # 4. SPECIAL TOKENS (показываем конкретные ID)
    print(f"4. SPECIAL TOKENS:")
    print(f"   - [CLS] (начало): {tokenizer.cls_token_id}")
    print(f"   - [SEP] (конец текста): {tokenizer.sep_token_id}")
    print(f"   - [PAD] (заполнитель): {tokenizer.pad_token_id}")

    # 5. ДЕМОНСТРАЦИЯ PADDING / TRUNCATION
    actual_len = len(tokenizer.encode(raw_text))
    print(f"\n5. МЕХАНИЗМЫ ОБРАБОТКИ:")
    print(f"   - Длина текста в токенах: {actual_len}")

    # Важное замечание: в вашем map() не было указано padding='max_length',
    # поэтому токены [PAD] появятся только если вы добавили их в tokenizer() или DataCollator.
    pad_count = example["input_ids"].count(tokenizer.pad_token_id)

    if pad_count > 0:
        print(f"   - PADDING: Добавлено {pad_count} токенов [PAD].")
    else:
        print(f"   - PADDING: В данном примере [PAD] не использовался (динамическая длина).")

    if actual_len > 128:
        print(f"   - TRUNCATION: Текст был обрезан до 128 токенов.")
    else:
        print(f"   - TRUNCATION: Не потребовался.")

    print("\n" + "-"*60 + "\n")

In [ ]:
# Data collator будет добавлять padding динамически, прямо при формировании батча.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

sample_batch = [tokenized_datasets["train"][i] for i in range(3)]
collated_batch = data_collator(sample_batch)

for key, value in collated_batch.items():
    print(f"{key}: shape={tuple(value.shape)}")

In [ ]:
# Загружаем модель для классификации.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

model.to(device)

print("Model class:", model.__class__.__name__)
print("Number of labels:", model.config.num_labels)
print("id2label:", model.config.id2label)

In [ ]:

# Функция метрик для Trainer.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
    }

In [ ]:
# Общие параметры обучения.
common_training_kwargs = dict(
    output_dir="outputs/s13_bert_finetuning_demo",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=2,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
)

try:
    training_args = TrainingArguments(
        evaluation_strategy="epoch",
        save_strategy="epoch",
        **common_training_kwargs,
    )
except TypeError:
    training_args = TrainingArguments(
        eval_strategy="epoch",
        save_strategy="epoch",
        **common_training_kwargs,
    )

training_args

In [ ]:
# Собираем Trainer и запускаем обучение.
# В transformers >= 5.0 аргумент tokenizer переименован в processing_class.
try:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
except TypeError:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

train_result = trainer.train()
train_result

In [ ]:
# История логов Trainer.
history_df = pd.DataFrame(trainer.state.log_history)
display(history_df.head(10))

plt.figure(figsize=(8, 4))

if "loss" in history_df.columns:
    train_logs = history_df.dropna(subset=["loss"])
    plt.plot(train_logs["step"], train_logs["loss"], marker="o", label="train loss")

if "eval_loss" in history_df.columns:
    eval_logs = history_df.dropna(subset=["eval_loss"])
    plt.plot(eval_logs["step"], eval_logs["eval_loss"], marker="s", label="eval loss")

plt.title("История обучения")
plt.xlabel("Шаг")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:

# В transformers >= 5.x NotebookProgressCallback теряет состояние после обучения,
# что вызывает RuntimeError при вызове evaluate() вне тренировочного цикла.
# Удаляем его перед standalone-оценкой – это стандартный обходной путь.
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

# Оценка Trainer на validation и test.
val_metrics = trainer.evaluate(tokenized_datasets["validation"])
test_metrics = trainer.evaluate(tokenized_datasets["test"])

print("Validation metrics:")
for k, v in val_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, (int, float)) else f"{k}: {v}")

print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, (int, float)) else f"{k}: {v}")

In [ ]:
import os
from pathlib import Path

ARTIFACT_PATH = Path('artifacts')
os.makedirs(ARTIFACT_PATH, exist_ok=True)

In [ ]:
# В transformers >= 5.x NotebookProgressCallback теряет состояние после обучения,
# что вызывает RuntimeError при вызове evaluate() вне тренировочного цикла.
# Удаляем его перед standalone-оценкой – это стандартный обходной путь.
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

# Оценка Trainer на validation и test.
val_metrics = trainer.evaluate(tokenized_datasets["validation"])
test_metrics = trainer.evaluate(tokenized_datasets["test"])

print("Validation metrics:")
for k, v in val_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, (int, float)) else f"{k}: {v}")

print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, (int, float)) else f"{k}: {v}")
# Детальные предсказания на тестовой выборке.
test_output = trainer.predict(tokenized_datasets["test"])
test_logits = test_output.predictions
test_preds = np.argmax(test_logits, axis=-1)
test_true = test_output.label_ids

print("Classification report on test:")
print(
    classification_report(
        test_true,
        test_preds,
        target_names=[id2label[i] for i in range(len(id2label))],
        zero_division=0,
    )
)

cm = confusion_matrix(test_true, test_preds)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm)

ax.set_xticks(range(len(label_names)))
ax.set_yticks(range(len(label_names)))
ax.set_xticklabels(label_names, rotation=30, ha="right")
ax.set_yticklabels(label_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar(im, ax=ax)
plt.tight_layout()
path = ARTIFACT_PATH / "confusion_matrix.png"
plt.savefig(path)
plt.show()

In [ ]:
import numpy as np

# 1. Получаем предсказания (если еще не сделали)
test_results = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(test_results.predictions, axis=-1)
labels = test_results.label_ids

# 2. Выводим 10 примеров
print(f"{'№':<3} | {'Результат':<10} | {'Текст (восстановлен из токенов)'}")
print("-" * 110)

for i in range(10):
    # Декодируем input_ids обратно в читаемый текст
    # skip_special_tokens=True уберет технические токены [CLS], [SEP], [PAD]
    decoded_text = tokenizer.decode(tokenized_datasets["test"][i]["input_ids"], skip_special_tokens=True)

    # Обрезаем для компактности
    display_text = (decoded_text[:85] + '..') if len(decoded_text) > 85 else decoded_text

    true_idx = labels[i]
    pred_idx = predictions[i]

    # Определяем статус
    status = "✅ OK" if true_idx == pred_idx else "❌ FAIL"

    # Получаем названия классов из конфигурации модели
    true_label = model.config.id2label[true_idx]
    pred_label = model.config.id2label[pred_idx]

    print(f"{i+1:<3} | {status:<10} | {display_text}")
    print(f"    └─ Реальный: {true_label} | Предсказано: {pred_label}\n")

Здесь мы видим классический «шум» в данных или очень тонкий контекст:

Почему модель выбрала anger (или fear): Слово «dismayed» (встревоженный, разочарованный) само по себе имеет негативную окраску. Модель сочла это проявлением агрессии или раздражения.

Почему в разметке joy? Это выглядит как ошибка разметки (label noise) в исходном датасете. Трудно представить ситуацию, где чувство тревоги и разочарования от одежды окружающих классифицируется как «радость».

Вывод: Модель здесь, возможно, «умнее» разметчика или датасет содержит сарказм, который не понятен без полного текста

In [ ]:
import pandas as pd

# 1. Выбираем первые 10 индексов (как в нашем выводе)
indices = range(10)

sample_data = []

for i in indices:
    # Декодируем текст из input_ids (без спецтокенов)
    text = tokenizer.decode(tokenized_datasets["test"][i]["input_ids"], skip_special_tokens=True)

    # Получаем ID меток из результатов предикта
    true_idx = labels[i]
    pred_idx = predictions[i]

    # Переводим ID в текстовые названия (joy, anger, fear и т.д.)
    true_label_name = model.config.id2label[true_idx]
    pred_label_name = model.config.id2label[pred_idx]

    sample_data.append({
        "text": text,
        "true_label": true_label_name,
        "pred_label": pred_label_name,
        "is_correct": true_idx == pred_idx
    })

# 2. Создаем DataFrame и сохраняем
df_10_samples = pd.DataFrame(sample_data)
df_10_samples.to_csv(ARTIFACT_PATH/"sample_predictions.csv", index=False, encoding="utf-8")

print("Файл 'sample_predictions.csv' с 10 примерами создан!")
# Выведем превью таблицы для проверки
print(df_10_samples[['true_label', 'pred_label', 'is_correct']])